# ng ∈ [6, 7] Optimization Results — 71nm PE, Y-even Parity

Loads saved data from `optimize_ng6to7.py` and plots:
1. Group index vs. wavelength for the top-5 configurations
2. Band diagrams with the slow-light window highlighted
3. Summary table of winning parameters

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

# Paths
NPZ_PATH  = 'output/optimize_ng6to7_results.npz'
JSON_PATH = 'output/optimize_ng6to7_results.json'

# Load
data = np.load(NPZ_PATH, allow_pickle=False)
with open(JSON_PATH) as f:
    summary = json.load(f)

# Constants
NG_LO, NG_HI = 6.0, 7.0
WL_TARGET = 1550.0
N_SIO2 = 1.44

print(f'Loaded {len(summary)} ranked configurations')
print(f'Top-5 band data keys: {[k for k in data.files if k.startswith("top")]}')

## 1. Summary Table — Top 10

In [ ]:
print(f'{"#":>3}  {"h_spine":>8}  {"h_rib":>7}  {"W_rib":>7}  '
      f'{"a_nm":>7}  {"band":>5}  {"λc(nm)":>8}  {"ng":>6}  {"σ(ng)":>7}  {"BW(nm)":>7}')
print('-' * 80)
for s in summary[:10]:
    print(f'{s["rank"]:>3}  {s["h_spine"]:>8.3f}  {s["h_rib"]:>7.3f}  {s["W_rib"]:>7.3f}  '
          f'{s["a_nm"]:>7.1f}  {s["band"]:>5}  {s["wl_center"]:>8.1f}  '
          f'{s["ng_mean"]:>6.2f}  {s["ng_std"]:>7.4f}  {s["bw_nm"]:>7.1f}')

## 2. Group Index vs. Wavelength — Top 5 Configurations

In [ ]:
def compute_ng(f_band, k_x):
    df_dk = np.gradient(f_band, k_x)
    df_dk = np.where(np.abs(df_dk) < 1e-12, np.nan, df_dk)
    return 1.0 / df_dk

fig, axes = plt.subplots(1, 1, figsize=(10, 6))
ax = axes

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
n_top = min(5, len(summary))

for i in range(n_top):
    s = summary[i]
    freqs_key = f'top{i}_all_freqs'
    kx_key    = f'top{i}_k_x'
    
    if freqs_key not in data.files:
        continue
    
    all_freqs = data[freqs_key]
    k_x       = data[kx_key]
    b         = s['band']
    a_nm      = s['a_nm']
    
    f_b = all_freqs[:, b]
    ng  = compute_ng(f_b, k_x)
    wl  = np.where(f_b > 0, a_nm / f_b, np.nan)
    
    # Only plot physical range
    mask = np.isfinite(ng) & (ng > 0) & (ng < 30) & (wl > 1400) & (wl < 1700)
    
    label = (f'#{s["rank"]} hs={s["h_spine"]:.3f} hr={s["h_rib"]:.3f} '
             f'Wr={s["W_rib"]:.3f} a={a_nm:.0f}nm\n'
             f'   ng={s["ng_mean"]:.2f}±{s["ng_std"]:.3f}, BW={s["bw_nm"]:.1f}nm')
    
    ax.plot(wl[mask], ng[mask], '-o', color=colors[i], lw=1.8, ms=3,
            label=label, alpha=0.85)

# Target window
ax.axhspan(NG_LO, NG_HI, color='lime', alpha=0.15, label=f'Target ng ∈ [{NG_LO}, {NG_HI}]')
ax.axhline(NG_LO, color='green', ls='--', lw=0.8)
ax.axhline(NG_HI, color='green', ls='--', lw=0.8)
ax.axvline(WL_TARGET, color='gray', ls=':', lw=1.0, label='1550 nm')

ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Group Index $n_g$', fontsize=12)
ax.set_title('Group Index vs. Wavelength — Top 5 Configs (y-even, 71nm PE)', fontsize=13)
ax.set_xlim(1450, 1650)
ax.set_ylim(3, 12)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('output/ng_vs_wavelength_top5.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: output/ng_vs_wavelength_top5.png')

## 3. Band Diagrams — Top 5

In [ ]:
fig, axes = plt.subplots(1, n_top, figsize=(5 * n_top, 5), sharey=True)
if n_top == 1:
    axes = [axes]

for i, ax in enumerate(axes[:n_top]):
    s = summary[i]
    freqs_key = f'top{i}_all_freqs'
    kx_key    = f'top{i}_k_x'
    if freqs_key not in data.files:
        continue
    
    all_freqs = data[freqs_key]
    k_x       = data[kx_key]
    b         = s['band']
    a_nm      = s['a_nm']
    nb        = all_freqs.shape[1]
    
    f_lo, f_hi = 0.25, 0.40
    f_1550 = a_nm / WL_TARGET
    
    # Light cone
    k_lc = np.linspace(k_x[0], k_x[-1], 100)
    f_lc = k_lc / N_SIO2
    ax.fill_between(k_lc, np.maximum(f_lc, f_lo), f_hi, alpha=0.08, color='gray')
    ax.plot(k_lc, f_lc, color='gray', lw=0.6)
    
    # All bands
    for bb in range(nb):
        ax.plot(k_x, all_freqs[:, bb], color='lightgray', lw=0.8)
    
    # Target band
    f_b = all_freqs[:, b]
    ax.plot(k_x, f_b, color='red', lw=2.0, label=f'Band {b}')
    
    # Slow-light window
    ng = compute_ng(f_b, k_x)
    wl = np.where(f_b > 0, a_nm / f_b, np.nan)
    sl = np.isfinite(ng) & (ng >= NG_LO) & (ng <= NG_HI) & (wl > 1400) & (wl < 1700)
    if np.any(sl):
        ax.axhspan(f_b[sl].min(), f_b[sl].max(), color='lime', alpha=0.20,
                   label=f'BW={s["bw_nm"]:.1f}nm')
    
    ax.axhline(f_1550, color='green', ls=':', lw=1.0, label='1550nm')
    ax.set_xlim(k_x[0], k_x[-1])
    ax.set_ylim(f_lo, f_hi)
    ax.set_xlabel('k (2π/a)')
    ax.set_title(f'#{s["rank"]} hs={s["h_spine"]:.3f} hr={s["h_rib"]:.3f}\n'
                 f'Wr={s["W_rib"]:.3f} a={a_nm:.0f}nm', fontsize=9)
    ax.legend(fontsize=7, loc='upper left')

axes[0].set_ylabel('Frequency (a/λ)')
fig.suptitle('Band Diagrams — Top 5 (y-even, target band highlighted)', fontsize=12)
plt.tight_layout()
plt.savefig('output/band_diagrams_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/band_diagrams_top5.png')

## 4. Winner Detail — Band Diagram + ng(λ)

In [ ]:
if len(summary) > 0 and 'top0_all_freqs' in data.files:
    s = summary[0]
    all_freqs = data['top0_all_freqs']
    k_x       = data['top0_k_x']
    b         = s['band']
    a_nm      = s['a_nm']
    nb        = all_freqs.shape[1]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 9),
                                    gridspec_kw={'height_ratios': [2, 1]})

    # -- Top panel: band diagram --
    f_lo, f_hi = 0.25, 0.40
    f_1550 = a_nm / WL_TARGET
    k_lc = np.linspace(k_x[0], k_x[-1], 100)
    f_lc = k_lc / N_SIO2
    ax1.fill_between(k_lc, np.maximum(f_lc, f_lo), f_hi, alpha=0.08, color='gray')
    ax1.plot(k_lc, f_lc, color='gray', lw=0.6)

    for bb in range(nb):
        ax1.plot(k_x, all_freqs[:, bb], color='lightgray', lw=0.8)

    f_b = all_freqs[:, b]
    ax1.plot(k_x, f_b, color='red', lw=2.5, label=f'Band {b} (TE0, y-even)')
    ax1.axhline(f_1550, color='green', ls=':', lw=1.0, label='1550 nm')

    ng = compute_ng(f_b, k_x)
    wl = np.where(f_b > 0, a_nm / f_b, np.nan)
    sl = np.isfinite(ng) & (ng >= NG_LO) & (ng <= NG_HI) & (wl > 1400) & (wl < 1700)
    if np.any(sl):
        ax1.axhspan(f_b[sl].min(), f_b[sl].max(), color='lime', alpha=0.20,
                    label=f'ng ∈ [{NG_LO},{NG_HI}], BW = {s["bw_nm"]:.1f} nm')

    ax1.set_xlim(k_x[0], k_x[-1])
    ax1.set_ylim(f_lo, f_hi)
    ax1.set_ylabel('Frequency (a/λ)', fontsize=12)
    ax1.set_title(
        f'BEST: a={a_nm:.1f}nm  h_spine={s["h_spine"]:.3f}  '
        f'W_rib={s["W_rib"]:.3f}  h_rib={s["h_rib"]:.3f}\n'
        f'ng={s["ng_mean"]:.2f}±{s["ng_std"]:.4f}  BW={s["bw_nm"]:.1f}nm  λc={s["wl_center"]:.1f}nm',
        fontsize=11)
    ax1.legend(fontsize=9, loc='upper left')
    ax1.tick_params(labelbottom=False)

    # -- Bottom panel: ng(λ) --
    mask = np.isfinite(ng) & (ng > 0) & (ng < 30) & (wl > 1400) & (wl < 1700)
    ax2.plot(wl[mask], ng[mask], 'r-o', lw=2.0, ms=4)
    ax2.axhspan(NG_LO, NG_HI, color='lime', alpha=0.15)
    ax2.axhline(NG_LO, color='green', ls='--', lw=0.8)
    ax2.axhline(NG_HI, color='green', ls='--', lw=0.8)
    ax2.axvline(WL_TARGET, color='gray', ls=':', lw=1.0)
    ax2.set_xlabel('Wavelength (nm)', fontsize=12)
    ax2.set_ylabel('Group Index $n_g$', fontsize=12)
    ax2.set_xlim(1450, 1650)
    ax2.set_ylim(3, 12)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_minor_locator(AutoMinorLocator())

    plt.tight_layout()
    plt.savefig('output/winner_band_ng.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: output/winner_band_ng.png')
else:
    print('No valid results to plot.')

## 5. Winning Configuration Summary

In [ ]:
if len(summary) > 0:
    w = summary[0]
    print('=' * 55)
    print('  WINNING CONFIGURATION')
    print('=' * 55)
    print(f'  a_nm    = {w["a_nm"]:.1f} nm')
    print(f'  h_spine = {w["h_spine"]:.4f} a  ({w["h_spine"]*w["a_nm"]:.1f} nm)')
    print(f'  W_rib   = {w["W_rib"]:.4f} a  ({w["W_rib"]*w["a_nm"]:.1f} nm)  [Wt = Wb]')
    print(f'  h_rib   = {w["h_rib"]:.4f} a  ({w["h_rib"]*w["a_nm"]:.1f} nm)  [ht = hb]')
    print(f'  t_PE    = 71.0 nm')
    print(f'  band    = {w["band"]}  (y-even / TE0 sector)')
    print(f'  ng_mean = {w["ng_mean"]:.3f}')
    print(f'  ng_std  = {w["ng_std"]:.4f}')
    print(f'  BW      = {w["bw_nm"]:.1f} nm  (ng ∈ [6, 7])')
    print(f'  λ_center = {w["wl_center"]:.1f} nm')
    print('=' * 55)
else:
    print('No valid configuration found.')